# Session 11 — Your turn: Experiments you can believe

Is a gain of 0.02 real? Answer it with a paired interval over seeds, then see how often a pile of null tweaks produces something that looks like a win.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np

# test losses of two setups trained with the same 12 seeds (lower is better)
seeds_a = np.array([3.0233, 3.0108, 2.9926, 3.0191, 3.0042, 3.0210, 3.0035, 3.0171, 3.0117, 3.0298, 3.0150, 3.0069])
seeds_b = np.array([2.9913, 2.9798, 2.9830, 2.9941, 2.9867, 3.0004, 2.9768, 2.9990, 2.9911, 3.0120, 2.9926, 2.9879])
from aij_check import check, close, summary

## Task 1: A paired bootstrap interval

For each seed take the difference `d = seeds_b - seeds_a`. Draw 5,000 bootstrap resamples of the 12 differences (`np.random.default_rng(0)`, sampling with replacement), compute each resample's mean, and store the 2.5th and 97.5th percentiles in `low` and `high`, the mean in `mean_diff`,
and `real` (True if the whole interval is below 0, meaning B is better).

In [2]:
d = seeds_b - seeds_a
gen = np.random.default_rng(0)
boot = np.array([d[gen.integers(0, len(d), len(d))].mean() for _ in range(5000)])
mean_diff = d.mean()
low, high = np.percentile(boot, 2.5), np.percentile(boot, 97.5)
real = bool(high < 0)
print(mean_diff, low, high, real)

-0.021691666666666665 -0.025100416666666715 -0.018357916666666634 True


In [3]:
check("B is lower by about 0.02 on average", lambda: close(mean_diff, -0.0200, 0.003))
check("the interval contains the mean", lambda: low < mean_diff < high)
check("the interval is narrow, about 0.01 wide", lambda: 0.005 < high - low < 0.02)
check("the whole interval is below zero: the gain is real", lambda: real is True)
summary()

PASS  B is lower by about 0.02 on average
PASS  the interval contains the mean
PASS  the interval is narrow, about 0.01 wide
PASS  the whole interval is below zero: the gain is real

4 of 4 checks passed.  Well done.


**Why it works.** Pairing by seed removes the part of the noise the two setups share, and the bootstrap turns the 12 differences into an interval without assuming a distribution. With one seed per setup the same 0.02 would be 1.5 standard errors, not enough.

## Task 2: Many tweaks, one winner

Without any simulation, work out the chance that **at least one** of `k` independent tests of a tweak that does nothing comes out 'significant' at level 0.05: `p_any(k) = 1 - 0.95**k`. Fill `chance` for k = 1, 5, 20 and 100, then write `bonferroni_level(k)` (the per-test level that keeps the chance of any false win at 0.05, which is `0.05 / k`)
and check `chance_corrected` for k = 20 using that level, `1 - (1 - bonferroni_level(20))**20`.

In [4]:
chance = {k: 1 - 0.95 ** k for k in (1, 5, 20, 100)}


def bonferroni_level(k):
    return 0.05 / k


chance_corrected = 1 - (1 - bonferroni_level(20)) ** 20
print(chance, chance_corrected)

{1: 0.050000000000000044, 5: 0.22621906250000023, 20: 0.6415140775914581, 100: 0.994079470779666} 0.04883012474683324


In [5]:
check("one test: 5%", lambda: close(chance[1], 0.05, 1e-9))
check("twenty tweaks: about 64% (the session's simulation with 3 seeds finds 28%, because its tests are not independent and use small samples)", lambda: close(chance[20], 0.6415, 0.001))
check("a hundred tweaks almost guarantee a false win", lambda: chance[100] > 0.99)
check("the corrected per-test level for 20 tests is 0.0025", lambda: close(bonferroni_level(20), 0.0025, 1e-12))
check("with the correction the chance stays near 5%", lambda: close(chance_corrected, 0.0488, 0.001))
summary()

PASS  one test: 5%
PASS  twenty tweaks: about 64% (the session's simulation with 3 seeds finds 28%, because its tests are not independent and use small samples)
PASS  a hundred tweaks almost guarantee a false win
PASS  the corrected per-test level for 20 tests is 0.0025
PASS  with the correction the chance stays near 5%

5 of 5 checks passed.  Well done.


**Why it works.** Every extra test is another ticket in the lottery. The formula assumes independent tests and ideal p-values; in the session's simulation the chance for twenty tweaks is 28% because three seeds per arm give a coarse t-test and the tweaks share one baseline. Either way the lesson is the same: say how many things you tried.